In [1]:
# ── CELL 1: Imports and GEE Initialisation ──────────────────────────────────
import ee
import geemap
import os

# Initialise GEE — replace with your actual project ID
ee.Initialize(project='ee-festac')

print("✓ GEE initialised successfully")
print(f"✓ Working directory: {os.getcwd()}")

/home/deysholey/.local/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


✓ GEE initialised successfully
✓ Working directory: /home/deysholey/GeoAID_Project/notebooks


In [2]:
# ── CELL 2: Load Administrative Boundaries (Three-Tier Study Area) ───────────
# Source: FAO GAUL 2015 — available natively in GEE, no download needed
# Level 0 = Country | Level 1 = State | Level 2 = LGA

# Nigeria (country boundary)
nigeria = ee.FeatureCollection("FAO/GAUL/2015/level1") \
            .filter(ee.Filter.eq('ADM0_NAME', 'Nigeria'))

# Lagos State (Tier 1 — conceptual study area)
lagos_state = ee.FeatureCollection("FAO/GAUL/2015/level1") \
                .filter(ee.Filter.eq('ADM0_NAME', 'Nigeria')) \
                .filter(ee.Filter.eq('ADM1_NAME', 'Lagos'))

# Amuwo Odofin LGA (Tier 2 — empirical pilot area)
amuwo_odofin = ee.FeatureCollection("FAO/GAUL/2015/level2") \
                 .filter(ee.Filter.eq('ADM0_NAME', 'Nigeria')) \
                 .filter(ee.Filter.eq('ADM1_NAME', 'Lagos')) \
                 .filter(ee.Filter.eq('ADM2_NAME', 'Amuwo Odofin'))

# Verify boundaries loaded correctly
print("Nigeria feature count:", nigeria.size().getInfo())
print("Lagos State feature count:", lagos_state.size().getInfo())
print("Amuwo Odofin feature count:", amuwo_odofin.size().getInfo())

Nigeria feature count: 37
Lagos State feature count: 1
Amuwo Odofin feature count: 1


In [3]:
# ── CELL 3: Interactive Map Visualisation ────────────────────────────────────

# Styling for each boundary tier
lagos_style = {
    'color': '2E75B6',      # blue outline
    'fillColor': '2E75B680', # semi-transparent blue fill
    'width': 2
}

amuwo_style = {
    'color': 'FF0000',       # red outline — pilot area stands out
    'fillColor': 'FF000040', # semi-transparent red fill
    'width': 3
}

# Get Amuwo Odofin centroid for map centering
amuwo_centroid = amuwo_odofin.geometry().centroid().coordinates().getInfo()
lon, lat = amuwo_centroid[0], amuwo_centroid[1]
print(f"Amuwo Odofin centroid — Lat: {lat:.4f}, Lon: {lon:.4f}")

# Build interactive map
Map = geemap.Map(center=[lat, lon], zoom=11)

# Add layers bottom to top
Map.addLayer(lagos_state.style(**lagos_style), {}, 'Lagos State (Tier 1 — Conceptual)')
Map.addLayer(amuwo_odofin.style(**amuwo_style), {}, 'Amuwo Odofin LGA (Tier 2 — Pilot)')

# Add layer control and display
Map.addLayerControl()
Map

Amuwo Odofin centroid — Lat: 6.4551, Lon: 3.2651


Map(center=[6.455061708555176, 3.2650782915242123], controls=(WidgetControl(options=['position', 'transparent_…

In [4]:
# ── CELL 4: Export Amuwo Odofin Boundary as GeoJSON ─────────────────────────
# This file is used in ALL subsequent notebooks as the master clip boundary

import json
import os

# Define output path
output_dir = os.path.expanduser("~/GeoAID_Project/data")
output_path = os.path.join(output_dir, "amuwo_odofin_boundary.geojson")

# Extract geometry and export
amuwo_geojson = amuwo_odofin.getInfo()

with open(output_path, 'w') as f:
    json.dump(amuwo_geojson, f, indent=2)

print(f"✓ Boundary exported: {output_path}")
print(f"✓ Features saved: {amuwo_geojson['features'][0]['properties']}")

✓ Boundary exported: /home/deysholey/GeoAID_Project/data/amuwo_odofin_boundary.geojson
✓ Features saved: {'ADM0_CODE': 182, 'ADM0_NAME': 'Nigeria', 'ADM1_CODE': 2230, 'ADM1_NAME': 'Lagos', 'ADM2_CODE': 191300, 'ADM2_NAME': 'Amuwo Odofin', 'DISP_AREA': 'NO', 'EXP2_YEAR': 3000, 'STATUS': 'Member State', 'STR2_YEAR': 1999, 'Shape_Area': 0.0143171843739, 'Shape_Leng': 0.593223938468}


In [5]:
# ── CELL 5: Session Summary ──────────────────────────────────────────────────
print("=" * 55)
print("  NOTEBOOK 01 — STUDY AREA DEFINITION: COMPLETE")
print("=" * 55)
print()
print("  Tier 1 (Conceptual): Lagos State ✓")
print("  Tier 2 (Empirical):  Amuwo Odofin LGA ✓")
print("  Boundary exported:   data/amuwo_odofin_boundary.geojson ✓")
print()
print("  Next: 02_dem_acquisition.ipynb")
print("        → Load SRTM DEM, clip to Amuwo Odofin,")
print("          derive elevation + slope + TWI")
print("=" * 55)

  NOTEBOOK 01 — STUDY AREA DEFINITION: COMPLETE

  Tier 1 (Conceptual): Lagos State ✓
  Tier 2 (Empirical):  Amuwo Odofin LGA ✓
  Boundary exported:   data/amuwo_odofin_boundary.geojson ✓

  Next: 02_dem_acquisition.ipynb
        → Load SRTM DEM, clip to Amuwo Odofin,
          derive elevation + slope + TWI
